In [1]:
import pandas as pd
import numpy as np
from sklift.datasets import fetch_x5


c:\Users\hai liang\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
##load in data
dataset = fetch_x5()

clients = dataset.data.clients
purchases = dataset.data.purchases
train = dataset.data.train

treatment = dataset.treatment
target = dataset.target

In [3]:
print("Clients shape:", clients.shape)
print("Purchases shape:", purchases.shape)
print("Train shape:", train.shape)

Clients shape: (400162, 5)
Purchases shape: (45786568, 13)
Train shape: (200039, 1)


In [4]:
##create a features table
features = train[["client_id"]].copy()

print(features.shape)
features.head()

(200039, 1)


,client_id
0,000012768d
1,000036f903
2,00010925a5
3,0001f552b0
4,00020e7b18


In [5]:
##check for invalid age
print(clients["age"].describe())

print("Age < 0:", (clients["age"] < 0).sum())
print("Age > 100:", (clients["age"] > 100).sum())

count    400162.000000
mean         46.488112
std          43.871218
min       -7491.000000
25%          34.000000
50%          45.000000
75%          59.000000
max        1901.000000
Name: age, dtype: float64
Age < 0: 96
Age > 100: 1049


In [6]:
#set valid age and put age to feature
valid_age = clients.loc[
    (clients["age"] >= 0) & (clients["age"] <= 100),
    "age"
]

median_age = valid_age.median()

print("Median valid age:", median_age)

client_age = clients[["client_id", "age"]].copy()

client_age.loc[
    (client_age["age"] < 0) | (client_age["age"] > 100),
    "age"
] = median_age

features = features.merge(
    client_age,
    on="client_id",
    how="left"
)

features.head()

print(features["age"].describe())
print("Invalid ages remaining:",
      ((features["age"] < 0) | (features["age"] > 100)).sum())

Median valid age: 45.0
count    200039.000000
mean         46.359500
std          15.909756
min           0.000000
25%          34.000000
50%          45.000000
75%          59.000000
max         100.000000
Name: age, dtype: float64
Invalid ages remaining: 0


In [7]:
##set gender

print(clients["gender"].value_counts(dropna=False))

gender_features = pd.get_dummies(
    clients[["client_id", "gender"]],
    columns=["gender"],
    prefix="gender",
    dtype=int
)

gender_features.head()

features = features.merge(
    gender_features,
    on="client_id",
    how="left"
)

features.head()

gender
U    185706
F    147649
M     66807
Name: count, dtype: int64


,client_id,age,gender_F,gender_M,gender_U
0,000012768d,45,0,0,1
1,000036f903,72,1,0,0
2,00010925a5,83,0,0,1
3,0001f552b0,33,1,0,0
4,00020e7b18,73,0,0,1


In [8]:
# number of transactions
num_transactions = (
    purchases
    .groupby("client_id")["transaction_id"]
    .nunique()
    .rename("num_transactions")
    .reset_index()
)

num_transactions.head()

features = features.merge(
    num_transactions,
    on="client_id",
    how="left"
)

features["num_transactions"].describe()

count    200039.000000
mean         20.120821
std          17.732130
min           1.000000
25%           8.000000
50%          15.000000
75%          27.000000
max         320.000000
Name: num_transactions, dtype: float64

In [9]:
#number of uniq products purchased
num_unique_products = (
    purchases
    .groupby("client_id")["product_id"]
    .nunique()
    .rename("num_unique_products")
    .reset_index()
)

num_unique_products.head()
features = features.merge(
    num_unique_products,
    on="client_id",
    how="left"
)

features["num_unique_products"].describe()

count    200039.000000
mean         75.287259
std          56.378792
min           1.000000
25%          35.000000
50%          62.000000
75%         101.000000
max        1364.000000
Name: num_unique_products, dtype: float64

In [10]:
##number of stores visited
num_stores_visited = (
    purchases
    .groupby("client_id")["store_id"]
    .nunique()
    .rename("num_stores_visited")
    .reset_index()
)

num_stores_visited.head()
features = features.merge(
    num_stores_visited,
    on="client_id",
    how="left"
)

features["num_stores_visited"].describe()

count    200039.000000
mean          2.920690
std           2.005216
min           1.000000
25%           2.000000
50%           2.000000
75%           4.000000
max          95.000000
Name: num_stores_visited, dtype: float64

In [11]:
##total number of products customer purchased
total_quantity = (
    purchases
    .groupby("client_id")["product_quantity"]
    .sum()
    .rename("total_quantity")
    .reset_index()
)

features = features.merge(
    total_quantity,
    on="client_id",
    how="left"
)

features["total_quantity"].describe()



count    200039.000000
mean        142.527997
std         140.550727
min           0.000000
25%          53.000000
50%         104.000000
75%         188.000000
max       10614.000000
Name: total_quantity, dtype: float64

In [12]:
##average item per transaction
features["avg_items_per_transaction"] = (
    features["total_quantity"] /
    features["num_transactions"]
)

features["avg_items_per_transaction"].describe()



count    200039.000000
mean          8.008086
std           5.421509
min           0.000000
25%           4.538462
50%           6.636364
75%           9.833333
max         120.000000
Name: avg_items_per_transaction, dtype: float64

In [13]:
##total,average,median and STD spending
transaction_spend = (
    purchases[
        ["client_id", "transaction_id", "purchase_sum"]
    ]
    .drop_duplicates(
        subset=["client_id", "transaction_id"]
    )
)

total_spend = (
    transaction_spend
    .groupby("client_id")["purchase_sum"]
    .sum()
    .rename("total_spend")
    .reset_index()
)

avg_transaction_spend = (
    transaction_spend
    .groupby("client_id")["purchase_sum"]
    .mean()
    .rename("avg_transaction_spend")
    .reset_index()
)

median_transaction_spend = (
    transaction_spend
    .groupby("client_id")["purchase_sum"]
    .median()
    .rename("median_transaction_spend")
    .reset_index()
)

spend_std = (
    transaction_spend
    .groupby("client_id")["purchase_sum"]
    .std()
    .rename("spend_std")
    .reset_index()
)

features = features.merge(total_spend, on="client_id", how="left")
features = features.merge(avg_transaction_spend, on="client_id", how="left")
features = features.merge(median_transaction_spend, on="client_id", how="left")
features = features.merge(spend_std, on="client_id", how="left")


In [14]:
##loyalty points bts
transaction_points = (
    purchases[
        [
            "client_id",
            "transaction_id",
            "regular_points_received",
            "regular_points_spent",
            "express_points_received",
            "express_points_spent"
        ]
    ]
    .drop_duplicates(
        subset=["client_id", "transaction_id"]
    )
)

transaction_points.head()


##total points earned for each customer

,client_id,transaction_id,regular_points_received,regular_points_spent,express_points_received,express_points_spent
0,000012768d,7e3e2e3984,10.0,0.0,0.0,0.0
19,000012768d,c1ca85d462,5.7,0.0,0.0,0.0
30,000012768d,6a0e96d0bc,8.0,0.0,0.0,0.0
46,000012768d,b34f23306e,2.0,0.0,0.0,0.0
52,000036f903,12b218b054,1.2,0.0,0.0,0.0


In [15]:
##total points earned for each customer
regular_points_received = (
    transaction_points
    .groupby("client_id")["regular_points_received"]
    .sum()
    .rename("regular_points_received")
    .reset_index()
)

features = features.merge(
    regular_points_received,
    on="client_id",
    how="left"
)

features["regular_points_received"].describe()


##how much points spent per customer

count    200039.000000
mean         78.022883
std          98.582251
min           0.000000
25%          23.300000
50%          49.200000
75%          96.700000
max        8635.000000
Name: regular_points_received, dtype: float64

In [16]:
##how much points spent per customer
regular_points_spent = (
    transaction_points
    .groupby("client_id")["regular_points_spent"]
    .sum()
    .rename("regular_points_spent")
    .reset_index()
)

features = features.merge(
    regular_points_spent,
    on="client_id",
    how="left"
)

features["regular_points_spent"].describe()




count    200039.000000
mean        -73.397088
std         132.516680
min      -10131.000000
25%         -98.000000
50%         -27.000000
75%           0.000000
max           0.000000
Name: regular_points_spent, dtype: float64

In [17]:
##total express points earned per customer
express_points_received = (
    transaction_points
    .groupby("client_id")["express_points_received"]
    .sum()
    .rename("express_points_received")
    .reset_index()
)

features = features.merge(
    express_points_received,
    on="client_id",
    how="left"
)

features["express_points_received"].describe()



##total express point spent per customer

count    200039.000000
mean          0.796720
std           7.655577
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         310.000000
Name: express_points_received, dtype: float64

In [18]:
##total express point spent per customer
express_points_spent = (
    transaction_points
    .groupby("client_id")["express_points_spent"]
    .sum()
    .rename("express_points_spent")
    .reset_index()
)

features = features.merge(
    express_points_spent,
    on="client_id",
    how="left"
)

features["express_points_spent"].describe()




count    200039.000000
mean         -6.454331
std          16.306043
min        -330.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           0.000000
Name: express_points_spent, dtype: float64

In [19]:
##loyalty point net
features["net_regular_points_change"] = (
    features["regular_points_received"]
    + features["regular_points_spent"]
)

features["net_regular_points_change"].describe()




count    200039.000000
mean          4.625795
std          89.682951
min       -4919.800000
25%         -12.200000
50%           9.200000
75%          33.100000
max        1553.600000
Name: net_regular_points_change, dtype: float64

In [20]:
##express point net
features["net_express_points_change"] = (
    features["express_points_received"]
    + features["express_points_spent"]
)

features["net_express_points_change"].describe()





count    200039.000000
mean         -5.657612
std          17.307266
min        -210.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         300.000000
Name: net_express_points_change, dtype: float64

In [21]:

##days since last purchase
last_purchase = (
    purchases
    .groupby("client_id")["transaction_datetime"]
    .max()
    .rename("last_purchase_date")
    .reset_index()
)

last_purchase["last_purchase_date"] = pd.to_datetime(
    last_purchase["last_purchase_date"]
)

reference_date = last_purchase["last_purchase_date"].max()

print("Reference date:", reference_date)

last_purchase["days_since_last_purchase"] = (
    reference_date
    - last_purchase["last_purchase_date"]
).dt.days

days_since_last_purchase = last_purchase[
    ["client_id", "days_since_last_purchase"]
]

features = features.merge(
    days_since_last_purchase,
    on="client_id",
    how="left"
)

features["days_since_last_purchase"].describe()

Reference date: 2019-03-18 23:40:03


count    200039.000000
mean          5.828853
std           5.668023
min           0.000000
25%           1.000000
50%           4.000000
75%           9.000000
max          22.000000
Name: days_since_last_purchase, dtype: float64

In [22]:
##average day between transactions
transaction_dates = (
    purchases[
        ["client_id", "transaction_id", "transaction_datetime"]
    ]
    .drop_duplicates(
        subset=["client_id", "transaction_id"]
    )
)

transaction_dates["transaction_datetime"] = pd.to_datetime(
    transaction_dates["transaction_datetime"]
)

transaction_dates = transaction_dates.sort_values(
    ["client_id", "transaction_datetime"]
)

transaction_dates.head()

transaction_dates["days_since_previous"] = (
    transaction_dates
    .groupby("client_id")["transaction_datetime"]
    .diff()
    .dt.total_seconds()
    / 86400
)

avg_days_between_transactions = (
    transaction_dates
    .groupby("client_id")["days_since_previous"]
    .mean()
    .rename("avg_days_between_transactions")
    .reset_index()
)

features = features.merge(
    avg_days_between_transactions,
    on="client_id",
    how="left"
)

features["avg_days_between_transactions"].describe()

count    196219.000000
mean          9.332346
std          10.487180
min           0.000012
25%           3.735622
50%           6.219644
75%          10.827138
max         113.002269
Name: avg_days_between_transactions, dtype: float64

In [23]:
##number of days btw first and last purchase
customer_activity_span = (
    transaction_dates
    .groupby("client_id")["transaction_datetime"]
    .agg(["min", "max"])
    .reset_index()
)

customer_activity_span["customer_activity_span"] = (
    customer_activity_span["max"]
    - customer_activity_span["min"]
).dt.days

customer_activity_span = customer_activity_span[
    ["client_id", "customer_activity_span"]
]

features = features.merge(
    customer_activity_span,
    on="client_id",
    how="left"
)

features["customer_activity_span"].describe()



##add treatment and target

count    200039.000000
mean         90.949660
std          28.373689
min           0.000000
25%          84.000000
50%         102.000000
75%         110.000000
max         116.000000
Name: customer_activity_span, dtype: float64

In [24]:
##add treatment and target
labels = pd.DataFrame({
    "client_id": train["client_id"].to_numpy(),
    "treatment": treatment.to_numpy(),
    "target": target.to_numpy()
})

labels.head()






,client_id,treatment,target
0,000012768d,0,1
1,000036f903,1,1
2,00010925a5,1,1
3,0001f552b0,1,1
4,00020e7b18,1,1


In [25]:
##merge
features = features.merge(
    labels,
    on="client_id",
    how="left",
    validate="one_to_one"
)

features.head()

,client_id,age,gender_F,gender_M,gender_U,num_transactions,num_unique_products,num_stores_visited,total_quantity,avg_items_per_transaction,...,regular_points_spent,express_points_received,express_points_spent,net_regular_points_change,net_express_points_change,days_since_last_purchase,avg_days_between_transactions,customer_activity_span,treatment,target
0,000012768d,45,0,0,1,4,46,3,54.0,13.500000,...,0.0,0.0,0.0,25.7,0.0,4,34.441906,103,0,1
1,000036f903,72,1,0,0,32,96,5,169.0,5.281250,...,0.0,60.0,0.0,54.9,60.0,1,3.515704,108,1,1
2,00010925a5,83,0,0,1,18,58,2,79.0,4.388889,...,-17.0,0.0,0.0,14.8,0.0,10,6.049572,102,1,1
3,0001f552b0,33,1,0,0,15,79,4,106.0,7.066667,...,0.0,0.0,0.0,78.9,0.0,2,8.010879,112,1,1
4,00020e7b18,73,0,0,1,18,175,4,394.0,21.888889,...,-592.0,0.0,-30.0,-305.9,-30.0,3,6.597343,112,1,1


In [26]:
print(features.shape)
print(features.columns.tolist())





(200039, 25)
['client_id', 'age', 'gender_F', 'gender_M', 'gender_U', 'num_transactions', 'num_unique_products', 'num_stores_visited', 'total_quantity', 'avg_items_per_transaction', 'total_spend', 'avg_transaction_spend', 'median_transaction_spend', 'spend_std', 'regular_points_received', 'regular_points_spent', 'express_points_received', 'express_points_spent', 'net_regular_points_change', 'net_express_points_change', 'days_since_last_purchase', 'avg_days_between_transactions', 'customer_activity_span', 'treatment', 'target']


In [27]:
#fill those with only 1 transaction  ( 0 for spend_std and avg between transaction)
features["spend_std"] = (
    features["spend_std"].fillna(0)
)

features["avg_days_between_transactions"] = (
    features["avg_days_between_transactions"].fillna(0)
)

In [28]:

features.to_csv(
    "retailhero_causal_forest_features.csv",
    index=False
)



